# Phase 3 — Multi-Scale Ensemble (W=30 v1 + W=60 v2)

**Goal**: keep v2 in-distribution strength (PR-AUC ≥ 0.90) while recovering
v1-level drift robustness by combining scores from both window sizes.

**Leak-free combination rules** (chosen on `cc1_val` FPR ≈ 1% only):
1. `and_gate` — alert only if **both** models exceed their own `val_p99`
2. `or_gate` — alert if **either** exceeds its `val_p99`
3. `max_z` — alert if max(z30, z60) exceeds a val-calibrated z-threshold
4. `mean_z` — alert if mean(z30, z60) exceeds a val-calibrated z-threshold

Where `z = (mse - mu_train) / sigma_train` using each model's own train stats.

**ID gate**: any selected rule that drops cc1_test PR-AUC below 0.90 or F1 below 0.88 is rejected.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle, joblib, os, copy, time
import matplotlib.pyplot as plt
from pathlib import Path
from collections import deque
from scipy import stats
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix,
    roc_auc_score, average_precision_score, precision_recall_curve,
)

def resolve_base():
    here = Path.cwd().resolve()
    candidates = [
        here if here.name == 'module3' else None,
        here.parent if here.name == 'module3_pipeline_v2' else None,
        Path(r'c:\Users\DELL\Documents\Claude\Projects\FYP\module3'),
        Path(r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'),
    ]
    for c in candidates:
        if c is not None and (c / 'models_v2').exists():
            return str(c)
    raise FileNotFoundError('Could not locate module3 BASE (models_v2 missing).')

BASE = resolve_base()
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
WIN_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1_v2')
WIN_DIR_V1 = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models_v2')
MODEL_DIR_V1 = os.path.join(BASE, 'models')
print('BASE =', BASE)

TARGET_FPR = 0.01
ALL_EVAL = ['cc1_test', 'drift_cc2']
print('Phase 3 constants ready.')


## Step 1 — Load both models


In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(),
            nn.Linear(hidden1, hidden2), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden2), nn.ReLU(),
            nn.Linear(hidden2, hidden1), nn.ReLU(),
            nn.Linear(hidden1, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)

    def decode(self, z):
        return self.decoder(z)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval()
        mu, _ = self.encode(x)
        return ((self.decode(mu) - x) ** 2).mean(dim=1)

def load_bundle(model_dir, win_dir, label):
    meta = pickle.load(open(os.path.join(model_dir, 'vae_cc1_meta.pkl'), 'rb'))
    ev = pickle.load(open(os.path.join(model_dir, 'vae_cc1_eval.pkl'), 'rb'))
    model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
    model.load_state_dict(torch.load(os.path.join(model_dir, 'vae_cc1.pt'), map_location='cpu'))
    model.eval()
    return {
        'label': label,
        'meta': meta,
        'eval': ev,
        'model': model,
        'clip': meta['clip'],
        'mu': float(meta['mu_train']),
        'sigma': float(meta['sigma_train']),
        'val_p99': float(ev['thresholds']['val_p99']),
        'win_dir': win_dir,
        'window_size': int(meta.get('window_size', 30 if 'v1' in label else 60)),
    }

v1 = load_bundle(MODEL_DIR_V1, WIN_DIR_V1, 'v1_w30')
v2 = load_bundle(MODEL_DIR, WIN_DIR, 'v2_w60')
print(f"v1: dim={v1['meta']['input_dim']} win={v1['window_size']} val_p99={v1['val_p99']:.5f}")
print(f"v2: dim={v2['meta']['input_dim']} win={v2['window_size']} val_p99={v2['val_p99']:.5f}")


## Step 2 — Score shared evaluation sets (align by cmdb_id + end timestamp)


In [ ]:
FEATURE_COLS = [
    'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes', 'container_memory_working_set_bytes',
    'container_memory_rss', 'container_memory_cache',
]
SPLIT_CSV = {
    'cc1_val': 'cc1_val.csv',
    'cc1_test': 'cc1_test.csv',
    'drift_cc2': 'drift_complex_case2.csv',
}

def build_windows(df, feature_cols, window_size, stride=1):
    rows = []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        data = g[feature_cols].values.astype(np.float32)
        is_gap = g['is_gap'].values
        labels = g['label'].values
        ts = g['timestamp'].values
        n = len(g)
        for i in range(0, n - window_size + 1, stride):
            if is_gap[i:i + window_size].any():
                continue
            rows.append({
                'cmdb_id': cmdb_id,
                'end_ts': int(ts[i + window_size - 1]),
                'y': int(labels[i:i + window_size].any()),
                'X': data[i:i + window_size],
            })
    return rows


def score_bundle(bundle, rows):
    pca = joblib.load(os.path.join(
        MODEL_DIR_V1 if bundle['label'].startswith('v1') else MODEL_DIR, 'cc1_pca.pkl'))['pca']
    X = np.stack([r['X'] for r in rows])
    Xp = np.clip(pca.transform(X.reshape(len(X), -1)), -bundle['clip'], bundle['clip']).astype(np.float32)
    with torch.no_grad():
        mse = bundle['model'].anomaly_score(torch.from_numpy(Xp)).numpy()
    keys = [(r['cmdb_id'], r['end_ts']) for r in rows]
    y = np.array([r['y'] for r in rows], dtype=np.int64)
    return keys, y, mse


def align_scores(keys_a, y_a, mse_a, keys_b, y_b, mse_b):
    """Inner-join on (cmdb_id, end_ts); both windows must end on the same timestamp."""
    map_b = {k: (yi, mi) for k, yi, mi in zip(keys_b, y_b, mse_b)}
    y, sa, sb = [], [], []
    for k, yi, mi in zip(keys_a, y_a, mse_a):
        if k in map_b:
            yj, mj = map_b[k]
            # labels should match (any-in-window over different lengths can differ — take OR)
            y.append(int(yi or yj))
            sa.append(mi)
            sb.append(mj)
    return np.array(y, dtype=np.int64), np.array(sa), np.array(sb)

aligned = {}
for name, fname in SPLIT_CSV.items():
    df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
    df['is_gap'] = df['is_gap'].astype(bool)
    rows30 = build_windows(df, FEATURE_COLS, v1['window_size'])
    rows60 = build_windows(df, FEATURE_COLS, v2['window_size'])
    k30, y30, m30 = score_bundle(v1, rows30)
    k60, y60, m60 = score_bundle(v2, rows60)
    y, s30, s60 = align_scores(k30, y30, m30, k60, y60, m60)
    aligned[name] = {'y': y, 'mse30': s30, 'mse60': s60}
    print(f'{name:10s}: aligned={len(y):,}  anom={int(y.sum())}  '
          f'(raw30={len(y30):,} raw60={len(y60):,})')


## Step 3 — Define combination rules + calibrate on cc1_val


In [ ]:
def zscore(mse, mu, sigma):
    return (mse - mu) / (sigma if sigma > 0 else 1.0)

def combine_scores(mse30, mse60, rule, z_thresh=None):
    z30 = zscore(mse30, v1['mu'], v1['sigma'])
    z60 = zscore(mse60, v2['mu'], v2['sigma'])
    if rule == 'and_gate':
        return ((mse30 > v1['val_p99']) & (mse60 > v2['val_p99'])).astype(int)
    if rule == 'or_gate':
        return ((mse30 > v1['val_p99']) | (mse60 > v2['val_p99'])).astype(int)
    if rule == 'max_z':
        return (np.maximum(z30, z60) > z_thresh).astype(int)
    if rule == 'mean_z':
        return (((z30 + z60) / 2.0) > z_thresh).astype(int)
    raise ValueError(rule)

def fpr_of(preds, y):
    # y is all-normal on cc1_val
    return float(preds.mean())

val = aligned['cc1_val']
assert val['y'].sum() == 0, 'cc1_val must be 100% normal'

# Fixed gates: measure val FPR
gate_val_fpr = {}
for rule in ['and_gate', 'or_gate']:
    preds = combine_scores(val['mse30'], val['mse60'], rule)
    gate_val_fpr[rule] = fpr_of(preds, val['y'])
    print(f'{rule:10s} val_FPR={gate_val_fpr[rule]*100:.2f}%')

# Calibrate z thresholds on val for target FPR
z_thresh = {}
for rule, score_fn in [
    ('max_z', lambda a, b: np.maximum(zscore(a, v1['mu'], v1['sigma']), zscore(b, v2['mu'], v2['sigma']))),
    ('mean_z', lambda a, b: 0.5 * (zscore(a, v1['mu'], v1['sigma']) + zscore(b, v2['mu'], v2['sigma']))),
]:
    scores = score_fn(val['mse30'], val['mse60'])
    thr = float(np.percentile(scores, (1 - TARGET_FPR) * 100))
    z_thresh[rule] = thr
    preds = (scores > thr).astype(int)
    print(f'{rule:10s} z_thresh={thr:.4f}  val_FPR={preds.mean()*100:.2f}%')


## Step 4 — Evaluate all rules + single-model baselines on test/drift


In [ ]:
def eval_preds(y, preds, scores_for_auc=None):
    out = {
        'precision': float(precision_score(y, preds, zero_division=0)),
        'recall': float(recall_score(y, preds, zero_division=0)),
        'f1': float(f1_score(y, preds, zero_division=0)),
    }
    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()
    out['fpr'] = float(fp / (fp + tn)) if (fp + tn) else float('nan')
    out['tp'], out['fp'], out['fn'], out['tn'] = int(tp), int(fp), int(fn), int(tn)
    if scores_for_auc is not None and y.sum() > 0 and (1 - y).sum() > 0:
        out['auc_pr'] = float(average_precision_score(y, scores_for_auc))
        out['auc_roc'] = float(roc_auc_score(y, scores_for_auc))
    else:
        out['auc_pr'] = out['auc_roc'] = float('nan')
    return out

results = {}
print(f'{"set":12s} {"method":14s} {"PR-AUC":>8s} {"F1":>7s} {"P":>7s} {"R":>7s} {"FPR":>8s}')

for name in ALL_EVAL:
    d = aligned[name]
    y, m30, m60 = d['y'], d['mse30'], d['mse60']
    results[name] = {}

    # Single-model baselines at their val_p99
    for label, mse, thr, mu, sig in [
        ('v1_alone', m30, v1['val_p99'], v1['mu'], v1['sigma']),
        ('v2_alone', m60, v2['val_p99'], v2['mu'], v2['sigma']),
    ]:
        preds = (mse > thr).astype(int)
        results[name][label] = eval_preds(y, preds, scores_for_auc=mse)
        r = results[name][label]
        print(f'{name:12s} {label:14s} {r["auc_pr"]:8.4f} {r["f1"]:7.3f} {r["precision"]:7.3f} '
              f'{r["recall"]:7.3f} {r["fpr"]:8.4f}')

    # Ensemble rules
    for rule in ['and_gate', 'or_gate', 'max_z', 'mean_z']:
        thr = z_thresh.get(rule)
        preds = combine_scores(m30, m60, rule, z_thresh=thr)
        # ranking score for PR-AUC: use the continuous combine
        if rule == 'and_gate':
            # soft-AND proxy: min of normalized scores
            score = np.minimum(zscore(m30, v1['mu'], v1['sigma']), zscore(m60, v2['mu'], v2['sigma']))
        elif rule == 'or_gate':
            score = np.maximum(zscore(m30, v1['mu'], v1['sigma']), zscore(m60, v2['mu'], v2['sigma']))
        elif rule == 'max_z':
            score = np.maximum(zscore(m30, v1['mu'], v1['sigma']), zscore(m60, v2['mu'], v2['sigma']))
        else:
            score = 0.5 * (zscore(m30, v1['mu'], v1['sigma']) + zscore(m60, v2['mu'], v2['sigma']))
        results[name][rule] = eval_preds(y, preds, scores_for_auc=score)
        r = results[name][rule]
        print(f'{name:12s} {rule:14s} {r["auc_pr"]:8.4f} {r["f1"]:7.3f} {r["precision"]:7.3f} '
              f'{r["recall"]:7.3f} {r["fpr"]:8.4f}')
    print()


## Step 5 — Select best rule under ID gate + drift objective


In [ ]:
CANDIDATES = ['and_gate', 'or_gate', 'max_z', 'mean_z']
id_gate = []
for rule in CANDIDATES:
    r = results['cc1_test'][rule]
    ok = (r['auc_pr'] >= 0.90) and (r['f1'] >= 0.88)
    id_gate.append((rule, ok, r))
    print(f'{rule:10s} ID_gate={"PASS" if ok else "FAIL"}  PR-AUC={r["auc_pr"]:.4f}  F1={r["f1"]:.3f}')

passed = [rule for rule, ok, _ in id_gate if ok]
if not passed:
    print('\\nNo ensemble rule passed the ID gate — fall back to v2_alone for ID, report best drift among all.')
    BEST_RULE = max(CANDIDATES, key=lambda r: results['drift_cc2'][r]['f1'])
    GATE_PASSED = False
else:
    # Among ID-safe rules, maximize drift F1, then precision, then minimize FPR
    BEST_RULE = max(
        passed,
        key=lambda r: (results['drift_cc2'][r]['f1'],
                       results['drift_cc2'][r]['precision'],
                       -results['drift_cc2'][r]['fpr']),
    )
    GATE_PASSED = True

print(f'\\nSelected BEST_RULE = {BEST_RULE}  (ID gate passed={GATE_PASSED})')
print('ID :', results['cc1_test'][BEST_RULE])
print('DRIFT:', results['drift_cc2'][BEST_RULE])


## Step 6 — Save


In [ ]:
save_results = {
    'phase': 3,
    'target_fpr': TARGET_FPR,
    'val_gate_fpr': gate_val_fpr,
    'z_thresh': z_thresh,
    'best_rule': BEST_RULE,
    'id_gate_passed': GATE_PASSED,
    'results': results,
    'v1_meta': {'val_p99': v1['val_p99'], 'mu': v1['mu'], 'sigma': v1['sigma'], 'window': v1['window_size']},
    'v2_meta': {'val_p99': v2['val_p99'], 'mu': v2['mu'], 'sigma': v2['sigma'], 'window': v2['window_size']},
}
out_path = os.path.join(MODEL_DIR, 'phase3_multiscale_eval.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')


## Phase 3 summary

Multi-scale fusion is the mechanism intended to break the W=30/W=60 Pareto frontier.
Selection is constrained by the ID gate so drift gains cannot silently destroy the
in-distribution win from window=60.
